In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os
import socket
from dataclasses import dataclass
from typing import Any

In [ ]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "datasets" 

In [ ]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

In [ ]:
import tiktoken 
import torch
import torch.nn as nn
from torch.nn import functional as F 
import torch.multiprocessing

In [ ]:
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

In [ ]:
if IS_PACE_SERVER:
    from model_training.framework.model.load_model.gpt2.configs import load_model_config
    from model_training.framework.model.load_model.gpt2.model import GPT
    from model_training.framework.model.components.inference.inference import gen_sample
    from model_training.framework.data.pre_train.tinyshakespeare.data import return_data, prepare_one_batch, DataLoaderLite
    from model_training.framework.train.tracking.logger import ExperimentLogger
    from model_training.framework.train.core.train import train_model
    from model_training.framework.train.core.config import TrainConfig
    from model_training.framework.train.utils.optimizer import configure_optimizer
    from model_training.framework.train.utils.lr_scheduler import cosine_lr
    from model_training.framework.train.core.multi_gpu import setup_ddp
else:
    ## Fix this import
    from model_training.framework.model.load_model.gpt2.model import GPT
        # from src.model import GPT

In [ ]:
ddp_details = setup_ddp()
print(ddp_details)
device = ddp_details['device']

In [ ]:
@dataclass
class TrainComponents:
    optimizer:Any
    dataloader:DataLoaderLite
    scheduler:Any

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
total_batch_size = 524288 ## 0.5 M (gpt3 paper)

In [ ]:
micro_batch_size= 2
seq_length = 1024

In [ ]:
## TO make sure this can run parallel, we just need to setup the right batch and configs, to ensure each process can run properly

assert total_batch_size % (micro_batch_size * seq_length * ddp_details['ddp_world_size']) ==0 , "Make sure total batch size is divisible"
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length * ddp_details['ddp_world_size'] )

In [ ]:
config = load_model_config(seq_length, "gpt2-small")
config.vocab_size = 50304           # Small Change to Vocab Size, Numbers in the power of 2 
config

In [ ]:
if ddp_details['master_process']:
    print(f"total desired batch size: {total_batch_size}")
    print(f"=> calculated gradient accumulation steps: {grad_accum_steps}")

#### Step-1:- Load the dataset

In [ ]:
dataloader = DataLoaderLite(batch_size=micro_batch_size, seq_len=seq_length, tokenizer=tokenizer, dataset_dir=ROOT_DATASET_DIR)

In [ ]:

len(dataloader.enc_data) // (dataloader.seq_len)

### Step-2:- Load the Model 

In [ ]:
model = GPT(config=config)
print(f"Model Loaded: {model}")
model.eval()            # Putting it in eval mode

### Step-3:- Training the Model 

In [ ]:

train_config = TrainConfig(num_steps=50, 
                        learning_rate=3e-4, 
                        total_batch_size= total_batch_size,
                        micro_batch_size =micro_batch_size, 
                        seq_length = seq_length, 
                        max_lr = 6e-4, 
                        min_lr =  6e-4 * 0.1,
                        warmup_steps= 715, 
                        accumulation_steps = grad_accum_steps,
                        # max_steps = 19073   # 19,073 steps is ~1 epoch, if data is 10B tokens and batch size 0.5M tokens
                        device=device)


In [ ]:
optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

In [ ]:
train_comp = TrainComponents(optimizer=optimizer , dataloader=dataloader, scheduler= cosine_lr)
train_comp

In [ ]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [ ]:
%time
### Training on Multiple Batches
model = train_model(model, train_components=train_comp, train_config=train_config)